# HAKE-MER — RoBERTa M2 mechanistic controls

**PFE — Mariem El Wedani** · SIIVA · Encadrant académique : Sahbi Bahroun · 2025–2026


Runs **both** variants on **RoBERTa-base** (3 seeds each):

1. `no_enc` — M1+M2 without inter-emotion Transformer
2. `no_xattn` — M1+M2 without phrase cross-attention

**Référence :** comparer les métriques test au chapitre 4 du rapport (même protocole : batch 16, LR 5e-5, 4 époques, trois graines).

After download: integrate zip → `python3 scripts/render_m2_controls_tex.py` → rebuild PDF.

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Runtime → Change runtime type → GPU")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import subprocess
from getpass import getpass
from pathlib import Path

import os
REPO = os.environ.get("GITHUB_REPO", "khalef-khalil/marii")
WORKDIR = Path("/content/marii")
PUBLIC_URL = f"https://github.com/{REPO}.git"
TRAIN_FLAGS = "--epochs 4 --batch-size 16 --lr 5e-5 --early-stopping-patience 0 --device cuda"
BACKBONE = "roberta-base"

def clone_repo() -> None:
    if WORKDIR.is_dir():
        subprocess.run(["git", "-C", str(WORKDIR), "fetch", "origin", "main"], check=True)
        subprocess.run(["git", "-C", str(WORKDIR), "reset", "--hard", "origin/main"], check=True)
        return
    r = subprocess.run(["git", "clone", "--depth", "1", PUBLIC_URL, str(WORKDIR)], capture_output=True)
    if r.returncode == 0:
        return
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = getpass("GitHub token: ")
    subprocess.run(["git", "clone", "--depth", "1", f"https://{token}@github.com/{REPO}.git", str(WORKDIR)], check=True)

clone_repo()
%cd {WORKDIR}
!git rev-parse --short HEAD

In [ ]:
!pip install -q -r requirements-train.txt

In [ ]:
!./run_m1_m2_controls_campaign.sh --variant no_enc --backbone {BACKBONE} {TRAIN_FLAGS}
!./run_m1_m2_controls_campaign.sh --variant no_xattn --backbone {BACKBONE} {TRAIN_FLAGS}

In [ ]:
import json
from pathlib import Path

slug = "roberta_base"
for variant in ("no_enc", "no_xattn"):
    path = Path(f"reference/artifacts/m1_m2_{variant}_{slug}_campaign.json")
    c = json.loads(path.read_text(encoding="utf-8"))
    m = c["test_aggregate"]["f1_macro"]
    print(path.name, f"F1-macro={m['mean']:.4f}±{m['std']:.4f}")

In [ ]:
import zipfile
from google.colab import files

slug = "roberta_base"
out_name = "roberta_m2_controls_campaign.zip"
zip_path = Path(f"/content/{out_name}")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for variant in ("no_enc", "no_xattn"):
        campaign = Path(f"reference/artifacts/m1_m2_{variant}_{slug}_campaign.json")
        zf.write(campaign, campaign.name)
        for m in sorted(Path("runs").glob(f"{slug}_seed*_m1_m2_{variant}/metrics.json")):
            zf.write(m, f"{m.parent.name}/{m.name}")
print(f"Download {out_name} ({zip_path.stat().st_size / 1e3:.1f} KB)")
files.download(str(zip_path))